# ybnote RL training (PPO) — Colab runner

Run cells top to bottom. Cell 2 will pop up two file-upload dialogs —
pick `code_for_colab.zip` first, then `charts_data.zip` (both from your
local `training/` folder). CPU runtime is fine (Runtime > Change runtime
type > CPU) — this workload is small unbatched per-step MLP forward
passes, a GPU would not help and adds per-call launch overhead.

In [ ]:
!pip -q install torch pandas numpy  # Colab usually already has these; harmless if already satisfied

In [ ]:
from google.colab import files
import zipfile, os

os.makedirs('/content/ybnote', exist_ok=True)
os.chdir('/content/ybnote')

print('Upload code_for_colab.zip:')
up1 = files.upload()
code_zip = list(up1.keys())[0]
with zipfile.ZipFile(code_zip) as z:
    z.extractall('.')

print('Upload charts_data.zip:')
up2 = files.upload()
data_zip = list(up2.keys())[0]
os.makedirs('output', exist_ok=True)
with zipfile.ZipFile(data_zip) as z:
    z.extractall('output')

print(os.listdir('.'))
print(os.listdir('training')[:5], '...')
print(len(os.listdir('output')), 'files in output/')

In [ ]:
# Sanity check: confirm chart loading works before committing to a long run
os.chdir('/content/ybnote/training')
!python -u train_rl.py --charts-dir ../output --holdout 5 --iterations 3 --rollout-steps 4096 --eval-every 3 --save /content/_smoketest.pt

In [ ]:
# The real run. ~800 iterations, checkpoints to rl_policy.pt whenever holdout
# hits improve (see train_rl.py) — safe to stop early and grab whatever the
# best checkpoint is so far. Expect roughly 15-20s/iteration on Colab CPU,
# i.e. a few hours total; Colab free-tier sessions can disconnect after long
# idle/runtime — reconnect and rerun this cell if that happens (rl_policy.pt
# only gets overwritten on a NEW best, so nothing is lost by a restart, but
# note this script itself doesn't resume iteration count — a restart starts
# a fresh 800-iteration run from the warm-start checkpoint again).
os.chdir('/content/ybnote/training')
!python -u train_rl.py --charts-dir ../output --holdout 5 --iterations 800 --rollout-steps 4096 --stage-b-after 500 --eval-every 20 --save rl_policy.pt

In [ ]:
from google.colab import files
files.download('/content/ybnote/training/rl_policy.pt')